# Chord Recognition Model Training

This notebook trains CNN models for chord classification using preprocessed CSV features.

**Goal:** Compare different feature types (Chroma, Hybrid, CQT60, CQT84) across two tasks:
- **Root-only classification:** 12 classes (C, Db, D, ..., B)
- **Root+Quality classification:** Full chord recognition (C:maj, D:min, G:dom, etc.)

**Feature Types:**
- **Chroma** (12 bins): Octave-invariant pitch class features
- **Hybrid** (36 bins): Low-CQT (24) + Chroma (12) - combines bass + harmony
- **CQT60** (60 bins): Reduced Constant-Q Transform (5 octaves)
- **CQT84** (84 bins): Full-range Constant-Q Transform (7 octaves)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Class

Custom PyTorch Dataset supporting both:
- **Root-only mode:** 12 classes (C, Db, D, ..., B)
- **Root+Quality mode:** Variable classes (C:maj, D:min, G:dom, etc.)

In [ ]:
class ChordDataset(Dataset):
    """Dataset for chord recognition from CSV features."""
    
    def __init__(self, csv_path, feature_type, label_mode='root'):
        """
        Args:
            csv_path: Path to CSV file with features
            feature_type: 'chroma', 'hybrid', 'cqt60', or 'cqt84'
            label_mode: 'root' (12 classes) or 'root_quality' (full chord)
        """
        self.df = pd.read_csv(csv_path)
        self.feature_type = feature_type
        self.label_mode = label_mode
        
        # Feature dimensions
        self.n_bins = {'chroma': 12, 'hybrid': 36, 'cqt60': 60, 'cqt84': 84}[feature_type]
        self.n_frames = 87
        
        # Build vocabulary based on label mode
        if label_mode == 'root':
            # Root-only: 12 classes
            self.label_to_idx = {
                'C': 0, 'Db': 1, 'D': 2, 'Eb': 3, 'E': 4, 'F': 5,
                'Gb': 6, 'G': 7, 'Ab': 8, 'A': 9, 'Bb': 10, 'B': 11
            }
        else:
            # Root+Quality: build from data
            unique_labels = sorted(self.df['label'].unique())
            self.label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        
        self.idx_to_label = {v: k for k, v in self.label_to_idx.items()}
        self.num_classes = len(self.label_to_idx)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Extract features (columns: feature_0, feature_1, ...)
        feature_cols = [col for col in self.df.columns if col.startswith('feature_')]
        features = row[feature_cols].values.astype(np.float32)
        
        # Reshape to (1, n_bins, n_frames) for CNN
        features = features.reshape(1, self.n_bins, self.n_frames)
        
        # Get label based on mode
        if self.label_mode == 'root':
            label_str = row['root']
        else:
            label_str = row['label']  # Full chord label (e.g., 'C:maj')
        
        label = self.label_to_idx[label_str]
        
        return torch.FloatTensor(features), torch.LongTensor([label]).squeeze()
    
    def get_sample_id(self, idx):
        """Get the sample ID for grouping."""
        return self.df.iloc[idx]['id']

## 3. Grouped Train/Val Split

**Critical:** We must group augmented versions of the same audio together to prevent data leakage.

Example:
- `beatrice_bar01.wav` and `beatrice_bar01_shift+2.wav` are the same audio
- Both must be in the same split (train OR val, never both)

In [ ]:
def create_grouped_dataloaders(dataset, batch_size=32, train_split=0.8, shuffle=True):
    """
    Create train/val dataloaders with grouped splitting to prevent data leakage.
    
    Args:
        dataset: ChordDataset instance
        batch_size: Batch size for training
        train_split: Fraction of data for training (0.8 = 80%)
        shuffle: Whether to shuffle the groups
    
    Returns:
        train_loader, val_loader
    """
    # Group indices by base name (remove _shift suffix)
    groups = defaultdict(list)
    
    for idx in range(len(dataset)):
        sample_id = dataset.get_sample_id(idx)
        
        # Remove augmentation suffixes to get base name
        base_name = sample_id
        if '_shift' in base_name:
            base_name = base_name.split('_shift')[0]
        if '_cycle' in base_name:
            base_name = base_name.split('_cycle')[0]
        
        groups[base_name].append(idx)
    
    # Shuffle groups (not individual samples!)
    group_list = list(groups.values())
    if shuffle:
        np.random.shuffle(group_list)
    
    # Split groups into train/val
    split_idx = int(len(group_list) * train_split)
    train_groups = group_list[:split_idx]
    val_groups = group_list[split_idx:]
    
    # Flatten to indices
    train_indices = [idx for group in train_groups for idx in group]
    val_indices = [idx for group in val_groups for idx in group]
    
    print(f"Total samples: {len(dataset)}")
    print(f"Total groups: {len(group_list)}")
    print(f"Number of classes: {dataset.num_classes}")
    print(f"Train samples: {len(train_indices)} ({len(train_indices)/len(dataset)*100:.1f}%)")
    print(f"Val samples: {len(val_indices)} ({len(val_indices)/len(dataset)*100:.1f}%)")
    
    # Create dataloaders
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader

## 4. Model Architecture

We use the **ChordCNN** architecture with:
- Vertical 5×1 kernel in first layer (captures harmonic structure)
- 4 convolutional layers with batch normalization
- Dropout for regularization
- Adaptive pooling (works with any input size)
- Flexible output layer (12 classes for root, more for root+quality)

This architecture will be used consistently across all feature types for fair comparison.

In [ ]:
class ChordCNN(nn.Module):
    """CNN for chord recognition."""
    
    def __init__(self, num_classes=12, input_bins=12, dropout=0.5):
        super().__init__()
        
        # Conv1: Vertical kernel (5x1) - captures harmonic structure
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(5, 1), padding=(2, 0))
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))
        
        # Conv2-4: Standard 3x3 kernels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3, 3), padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        # Adaptive pooling handles different input sizes
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Linear(256, 256)
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # Input: (batch, 1, n_bins, 87)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.adaptive_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        
        return x


def count_parameters(model):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Training Function

Implements:
- Training loop with validation
- Early stopping (patience-based)
- Learning rate scheduling (ReduceLROnPlateau)
- Loss and accuracy tracking

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=30, lr=0.001, patience=5):
    """
    Train a model with early stopping and learning rate scheduling.
    
    Args:
        model: PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum epochs to train
        lr: Initial learning rate
        patience: Early stopping patience
    
    Returns:
        history: Dict with training metrics
    """
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'epoch_times': []
    }
    
    best_val_acc = 0.0
    patience_counter = 0
    
    print(f"\nTraining for up to {num_epochs} epochs...")
    print(f"Early stopping patience: {patience}")
    print("-" * 70)
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
        
        # Calculate metrics
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        epoch_time = time.time() - epoch_start
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)
        
        # Print progress
        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"Time: {epoch_time:.1f}s")
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                print(f"Best validation accuracy: {best_val_acc:.2f}%")
                break
    
    print("-" * 70)
    print(f"Training completed!")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Average epoch time: {np.mean(history['epoch_times']):.1f}s")
    
    return history

## 6. Visualization Functions

In [ ]:
def plot_feature_comparison(histories, labels, task_name="Root Classification"):
    """
    Plot training curves comparing different feature types.
    
    Args:
        histories: List of history dicts
        labels: List of feature type names
        task_name: Name of the task (e.g., 'Root Classification')
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Define colors for each feature type
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    # Loss curves
    ax = axes[0]
    for history, label, color in zip(histories, labels, colors[:len(labels)]):
        epochs = range(1, len(history['train_loss']) + 1)
        ax.plot(epochs, history['train_loss'], '--', alpha=0.4, color=color)
        ax.plot(epochs, history['val_loss'], '-', linewidth=2.5, color=color, label=label)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title('Loss Curves (solid = validation, dashed = training)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax = axes[1]
    for history, label, color in zip(histories, labels, colors[:len(labels)]):
        epochs = range(1, len(history['train_acc']) + 1)
        ax.plot(epochs, history['train_acc'], '--', alpha=0.4, color=color)
        ax.plot(epochs, history['val_acc'], '-', linewidth=2.5, color=color, label=label)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title('Accuracy Curves (solid = validation, dashed = training)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Feature Type Comparison: {task_name}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save with task-specific filename
    filename = f"feature_comparison_{task_name.lower().replace(' ', '_').replace('+', '_')}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()


def create_comparison_report(results, task_name="Root"):
    """
    Create a comparison table of different feature types.
    
    Args:
        results: List of dicts with keys: 'name', 'history', 'params', 'num_classes'
        task_name: Name of the task
    
    Returns:
        DataFrame with comparison metrics
    """
    comparison_data = []
    
    for result in results:
        history = result['history']
        
        comparison_data.append({
            'Task': task_name,
            'Feature Type': result['name'],
            'Input Bins': result['n_bins'],
            'Num Classes': result['num_classes'],
            'Best Val Acc (%)': f"{max(history['val_acc']):.2f}",
            'Final Val Acc (%)': f"{history['val_acc'][-1]:.2f}",
            'Final Train Acc (%)': f"{history['train_acc'][-1]:.2f}",
            'Overfitting Gap (%)': f"{history['train_acc'][-1] - history['val_acc'][-1]:.2f}",
            'Epochs': len(history['train_loss']),
            'Time (min)': f"{sum(history['epoch_times'])/60:.1f}"
        })
    
    df = pd.DataFrame(comparison_data)
    return df

## 7. Experiment Configuration

**Instructions for Kaggle:**
1. Upload your CSV files as a Kaggle Dataset
2. Add the dataset to this notebook
3. Update the paths below to point to your dataset
4. Choose which experiments to run

In [ ]:
# Configuration
CONFIG = {
    # Paths (update these for Kaggle)
    'chroma_csv': '/kaggle/input/your-dataset/chroma_features.csv',
    'hybrid_csv': '/kaggle/input/your-dataset/hybrid_features.csv',
    'cqt60_csv': '/kaggle/input/your-dataset/cqt60_features.csv',
    'cqt84_csv': '/kaggle/input/your-dataset/cqt84_features.csv',
    
    # Training parameters
    'batch_size': 32,
    'num_epochs': 30,
    'learning_rate': 0.001,
    'patience': 5,
    'train_split': 0.8,
    'dropout': 0.5,
    
    # Tasks to run
    'run_root_task': True,           # Root-only classification (12 classes)
    'run_root_quality_task': True,   # Full chord classification (root+quality)
    
    # Feature types to compare (set to True to enable)
    'run_chroma': True,      # Recommended: octave-invariant
    'run_hybrid': True,      # Recommended: best overall expected
    'run_cqt60': True,       # Optional: more frequency detail
    'run_cqt84': False,      # Optional: full range (slowest)
}

print("Configuration:")
print(f"  Model: ChordCNN (4 conv layers + batch norm + dropout)")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Max epochs: {CONFIG['num_epochs']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Early stopping patience: {CONFIG['patience']}")
print(f"  Train/Val split: {CONFIG['train_split']:.0%}/{1-CONFIG['train_split']:.0%}")
print(f"\nTasks:")
print(f"  Root-only: {CONFIG['run_root_task']}")
print(f"  Root+Quality: {CONFIG['run_root_quality_task']}")

## 8. Run Experiments

Train models on both tasks (root-only and root+quality) across different feature types.

In [ ]:
# Store results for both tasks
root_results = []
root_quality_results = []

# Define feature configurations
feature_configs = []
if CONFIG['run_chroma']:
    feature_configs.append(('chroma', CONFIG['chroma_csv'], 12, 'Chroma'))
if CONFIG['run_hybrid']:
    feature_configs.append(('hybrid', CONFIG['hybrid_csv'], 36, 'Hybrid'))
if CONFIG['run_cqt60']:
    feature_configs.append(('cqt60', CONFIG['cqt60_csv'], 60, 'CQT60'))
if CONFIG['run_cqt84']:
    feature_configs.append(('cqt84', CONFIG['cqt84_csv'], 84, 'CQT84'))

# Run experiments
experiment_num = 0

for feature_type, csv_path, n_bins, feature_name in feature_configs:
    
    # Task 1: Root-only classification
    if CONFIG['run_root_task']:
        experiment_num += 1
        print("\n" + "=" * 80)
        print(f"EXPERIMENT {experiment_num}: {feature_name.upper()} - ROOT CLASSIFICATION (12 classes)")
        print("=" * 80)
        
        dataset = ChordDataset(csv_path, feature_type, label_mode='root')
        train_loader, val_loader = create_grouped_dataloaders(
            dataset, 
            batch_size=CONFIG['batch_size'],
            train_split=CONFIG['train_split']
        )
        
        model = ChordCNN(num_classes=dataset.num_classes, input_bins=n_bins, dropout=CONFIG['dropout'])
        print(f"\nModel: ChordCNN")
        print(f"Parameters: {count_parameters(model):,}")
        
        history = train_model(
            model, train_loader, val_loader,
            num_epochs=CONFIG['num_epochs'],
            lr=CONFIG['learning_rate'],
            patience=CONFIG['patience']
        )
        
        root_results.append({
            'name': feature_name,
            'n_bins': n_bins,
            'num_classes': dataset.num_classes,
            'history': history,
            'params': count_parameters(model)
        })
    
    # Task 2: Root+Quality classification
    if CONFIG['run_root_quality_task']:
        experiment_num += 1
        print("\n" + "=" * 80)
        print(f"EXPERIMENT {experiment_num}: {feature_name.upper()} - ROOT+QUALITY CLASSIFICATION")
        print("=" * 80)
        
        dataset = ChordDataset(csv_path, feature_type, label_mode='root_quality')
        train_loader, val_loader = create_grouped_dataloaders(
            dataset,
            batch_size=CONFIG['batch_size'],
            train_split=CONFIG['train_split']
        )
        
        model = ChordCNN(num_classes=dataset.num_classes, input_bins=n_bins, dropout=CONFIG['dropout'])
        print(f"\nModel: ChordCNN")
        print(f"Parameters: {count_parameters(model):,}")
        
        history = train_model(
            model, train_loader, val_loader,
            num_epochs=CONFIG['num_epochs'],
            lr=CONFIG['learning_rate'],
            patience=CONFIG['patience']
        )
        
        root_quality_results.append({
            'name': feature_name,
            'n_bins': n_bins,
            'num_classes': dataset.num_classes,
            'history': history,
            'params': count_parameters(model)
        })

## 9. Results Visualization and Report

In [ ]:
# Plot Root-only task results
if len(root_results) > 0:
    print("\n" + "#" * 80)
    print("ROOT CLASSIFICATION RESULTS (12 classes)")
    print("#" * 80)
    
    histories = [r['history'] for r in root_results]
    labels = [r['name'] for r in root_results]
    
    plot_feature_comparison(histories, labels, task_name="Root Classification")

In [ ]:
# Plot Root+Quality task results
if len(root_quality_results) > 0:
    print("\n" + "#" * 80)
    print("ROOT+QUALITY CLASSIFICATION RESULTS")
    print("#" * 80)
    
    histories = [r['history'] for r in root_quality_results]
    labels = [r['name'] for r in root_quality_results]
    
    plot_feature_comparison(histories, labels, task_name="Root+Quality Classification")

In [ ]:
# Create comprehensive comparison report
all_reports = []

if len(root_results) > 0:
    root_df = create_comparison_report(root_results, task_name="Root-only")
    all_reports.append(root_df)

if len(root_quality_results) > 0:
    root_quality_df = create_comparison_report(root_quality_results, task_name="Root+Quality")
    all_reports.append(root_quality_df)

if len(all_reports) > 0:
    combined_df = pd.concat(all_reports, ignore_index=True)
    
    print("\n" + "=" * 90)
    print("COMPREHENSIVE COMPARISON REPORT")
    print("=" * 90)
    print(combined_df.to_string(index=False))
    
    # Find best for each task
    for task in combined_df['Task'].unique():
        task_df = combined_df[combined_df['Task'] == task]
        best_idx = task_df['Best Val Acc (%)'].astype(str).str.rstrip('%').astype(float).argmax()
        best_feature = task_df.iloc[best_idx]['Feature Type']
        best_acc = task_df.iloc[best_idx]['Best Val Acc (%)']
        num_classes = task_df.iloc[best_idx]['Num Classes']
        
        print("\n" + "-" * 90)
        print(f"BEST for {task} ({num_classes} classes): {best_feature} with {best_acc}% validation accuracy")
        print("-" * 90)
    
    # Save report to CSV
    combined_df.to_csv('comprehensive_comparison_report.csv', index=False)
    print("\nReport saved to: comprehensive_comparison_report.csv")
else:
    print("No experiments were run. Enable tasks and feature types in CONFIG.")

## 10. Conclusions and Next Steps

### Expected Results:

**Root-only task (12 classes):**
- Target: 40-60% accuracy (baseline: 8.3% random)
- Chroma features should perform well (octave-invariant)
- Hybrid likely best overall

**Root+Quality task (50-80 classes):**
- Target: 20-40% accuracy (baseline: 1-2% random)
- Harder task, needs more harmonic information
- CQT features may help distinguish chord quality

### Key Findings to Look For:
1. **Task difficulty**: How much harder is root+quality vs root-only?
2. **Feature effectiveness**: Do different features excel at different tasks?
3. **Overfitting**: Does the harder task overfit more?

### If accuracy is low:
- Root-only <35%: Increase epochs, reduce dropout, check data leakage
- Root+Quality <15%: Expected! Try longer training, lower learning rate

### Next Steps:
1. **Choose best feature per task** based on validation accuracy
2. **Analyze confusion matrices** (which chords/qualities are hardest?)
3. **Consider hierarchical approach**: Predict root first, then quality
4. **Try ensemble methods**: Combine multiple feature types